# Egyptian dental clinic voice — training

## What changed in this version

**1. NileTTS is gone.** Replaced with the Kaggle dataset
`ahmedshafiq12/egyptian-audio-dataset-collected-from-youtube`, attached through **Add Input**.
That removes the entire failure surface that dominated earlier runs: no Hub download, no
rate limits, no xet errors, no 20 GB disk crisis, no chunked resume logic. The files are
just mounted read-only at `/kaggle/input/` and read in place.

Be aware what that dataset is: it was built for **speech-to-text** by pulling audio out of
YouTube videos and using the subtitles as labels. So expect many speakers, background noise
and music, and subtitle text that is often approximate rather than verbatim. It is useful for
Egyptian dialect coverage; it is not clean studio speech. Quality filters below drop the worst
of it, and `EXTRA_MAX_CLIPS` keeps it from swamping your own recordings.

**2. Your corrected 600.** Every recording's filename now equals its spreadsheet ID, so
matching is direct. The old category-prefix workaround is kept only as a fallback in case you
point this at the older, unfixed folder.

**3. Trim fixed.** `TRIM_TOP_DB` 30 → 40 and edge padding 0.05s → 0.15s. The old threshold cut
into the gradual onset ramp at the start of each clip, which is the most likely cause of the
first-word-garbled behaviour in the last run.

**4. Reference-conditioned generation.** The test cell now supplies real global (speaker) tokens
from one of your recordings instead of making the model invent them. This is how Spark-TTS
voice cloning is meant to work.

**Worth knowing before you run:** the last training run used data where 519 of 598 recordings
were paired with the *wrong* transcript, because of the recording-session filename shift. That
alone could explain the babbling. This run is the first with correct pairs throughout, so
judge it fresh rather than against the last one.

**One mandatory kernel restart**, marked below.

### Inputs to attach (Add Input, right-hand panel)
1. your recordings + spreadsheet dataset
2. `ahmedshafiq12/egyptian-audio-dataset-collected-from-youtube`

In [ ]:
# ============================================================
# CONFIG
# ============================================================

HF_USERNAME = "nour21k"

ARABIC_BASE_REPO    = "MrEzzat/Spark_TTS_Arabic"
ARABIC_SNAPSHOT_DIR = "Spark-TTS-Arabic"
SPARK_BASE_REPO     = "SparkAudio/Spark-TTS-0.5B"
SPARK_SNAPSHOT_DIR  = "Spark-TTS-0.5B"

# --- your data ---
SPREADSHEET_NAME    = "Egyptian_Dental_TTS_Dataset_600_MATCHED.xlsx"
SPREADSHEET_SHEET   = "Recording Script"
RECORDINGS_DIR_NAME = "fixed600"      # folder holding the corrected 600 wavs
OWN_DATASET_HINT    = "record"        # substring of YOUR dataset's folder name
EXTRA_ROOT_EXACT     = ("/kaggle/input/datasets/ahmedshafiq12/"
                        "egyptian-audio-dataset-collected-from-youtube/dataset")
PREPPED_DIR         = "prepped_wavs"

# --- audio preprocessing ---
TRIM_TOP_DB = 40          # was 30; 30 cut into the onset of the first word
EDGE_PAD_SEC = 0.15       # was 0.05
CODEC_MIN_SCORE = 0.35

# --- Egyptian dialect data (Kaggle dataset, replaces NileTTS) ---
USE_EXTRA_DIALECT = True
EXTRA_DATASET_HINT = "egyptian-audio"     # substring used to find it under /kaggle/input
EXTRA_MAX_CLIPS = 1200
EXTRA_MIN_SEC   = 1.5     # drop fragments
EXTRA_MAX_SEC   = 12.0    # drop long clips that would blow past MAX_SEQ_LEN
EXTRA_MIN_CHARS = 8       # drop near-empty subtitle lines
OWN_OVERSAMPLE  = 3       # repeat your clips so your voice is not outnumbered

# --- files ---
OWN_JSONL   = "dental_tokenized.jsonl"
EXTRA_JSONL = "extra_dialect_tokenized.jsonl"

# --- outputs ---
OUTPUT_MODEL_ID = f"{HF_USERNAME}/spark-tts-egyptian-dental-clinic-voice-v3"

# --- training ---
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
GRAD_ACCUM = 4
MAX_SEQ_LEN = 2048
MAX_GEN_TOKENS = 700

TEST_SENTENCES = [
    "افتح بقك من فضلك",
    "هرجع أشوفك الأسبوع الجاي",
    "الحشو محتاج جلستين بس",
]

print("Config loaded.")
print("Dialect data:", "Kaggle Egyptian YouTube set" if USE_EXTRA_DIALECT else "none (your voice only)")
print("Output:", OUTPUT_MODEL_ID)

In [ ]:
import glob, shutil, os

HUB_PIN = "huggingface_hub>=0.34.0,<1.0"

# rmtree, not `pip uninstall`: uninstall only deletes files listed in a package's own
# manifest, so files left by a different transformers version survive it. That is what
# kept resurrecting "cannot import name ... from transformers.utils".
for _p in glob.glob("/usr/local/lib/python3.12/dist-packages/transformers*"):
    print("removing", _p)
    shutil.rmtree(_p, ignore_errors=True)

!pip install --no-cache-dir transformers==4.56.2

# --no-deps: current peft releases can require transformers>=5, which would silently
# undo the pin above.
!pip install --no-deps --no-cache-dir peft

# Kaggle ships torchao 0.10.0 and transformers 4.56.2 refuses it on import.
!pip uninstall -y torchao

!pip install accelerate safetensors datasets
!pip install omegaconf einx einops soundfile librosa openpyxl pandas pyarrow

!pip uninstall -y huggingface_hub
!pip install --no-cache-dir "{HUB_PIN}"

if not os.path.exists("Spark-TTS/sparktts"):
    !git clone https://github.com/SparkAudio/Spark-TTS
else:
    print("Spark-TTS repo already present.")

print("\nINSTALL DONE - RESTART THE KERNEL NOW, then continue from the next cell.")

## RESTART THE KERNEL HERE

**Run → Restart & clear cell outputs**, then continue below. Do not re-run the install cell.

The transformers reinstall only takes effect in a fresh Python process. Every "same error again"
in this project traced back to skipping this.

In [ ]:
import glob, os
import importlib.metadata as md

dist_infos = glob.glob("/usr/local/lib/python3.12/dist-packages/transformers-*.dist-info")
print("transformers dist-info dirs:", [os.path.basename(d) for d in dist_infos])
if len(dist_infos) > 1:
    raise RuntimeError("More than one transformers version installed. Re-run the install cell.")

hub_ver = md.version("huggingface_hub")
print("huggingface_hub", hub_ver)
if int(hub_ver.split(".")[0]) >= 1:
    raise RuntimeError(f"huggingface_hub {hub_ver} installed, but transformers 4.56.2 needs <1.0.")

import torch, transformers, peft, librosa, soundfile, pandas
from transformers import AutoModelForCausalLM, AutoTokenizer

meta_ver = md.version("transformers")
print("transformers __version__:", transformers.__version__, "| dist-info:", meta_ver)
if transformers.__version__ != meta_ver:
    raise RuntimeError("transformers files come from mixed versions. Re-run the install cell.")

print("peft", peft.__version__, "| torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("All imports clean.")

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
HF_TOKEN = HF_TOKEN.strip().replace('"', '').replace("'", "")
if HF_TOKEN.lower().startswith("bearer "):
    HF_TOKEN = HF_TOKEN.split(" ", 1)[1].strip()
if not HF_TOKEN.startswith("hf_"):
    raise ValueError("HF_TOKEN is invalid - it should start with 'hf_'.")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Logged in as:", whoami(token=HF_TOKEN)["name"])

## Locate your data

In [ ]:
import os, glob

# Resolve BOTH datasets here, so they can never be confused for each other.
# The YouTube dialect set contains far more wav files than your 600, so a naive
# "folder with the most wavs" search would pick IT as your recordings folder.
attached = sorted(glob.glob("/kaggle/input/*"))
print("attached datasets:")
for d in attached:
    print("   ", os.path.basename(d))

if os.path.isdir(EXTRA_ROOT_EXACT):
    EXTRA_ROOT = EXTRA_ROOT_EXACT
else:
    # mount path can differ by session/Kaggle version - fall back to a name search
    EXTRA_ROOT = next((d for d in attached
                       if EXTRA_DATASET_HINT in os.path.basename(d).lower()), None)
    if EXTRA_ROOT is None:
        for root, dirs, files in os.walk("/kaggle/input"):
            if "egyptian-audio-dataset" in root.lower():
                EXTRA_ROOT = root
                break
if USE_EXTRA_DIALECT and EXTRA_ROOT is None:
    raise FileNotFoundError(
        "Could not find the Egyptian YouTube dataset at the expected path "
        f"({EXTRA_ROOT_EXACT}) or by name search. Attach it via Add Input, or update "
        "EXTRA_ROOT_EXACT in CONFIG to match what /kaggle/input actually shows."
    )

# everything that is NOT the dialect set is a candidate for your own recordings
own_roots = [d for d in attached if d != EXTRA_ROOT]

def find_in(roots, name):
    for r in roots:
        hits = glob.glob(f"{r}/**/{name}", recursive=True)
        if hits:
            return hits[0]
    return None

SPREADSHEET_PATH = find_in(own_roots, SPREADSHEET_NAME)
if SPREADSHEET_PATH is None:
    for r in own_roots:
        xl = sorted(glob.glob(f"{r}/**/*.xlsx", recursive=True))
        if xl:
            SPREADSHEET_PATH = xl[0]
            break

RECORDINGS_FOLDER = find_in(own_roots, RECORDINGS_DIR_NAME)
if RECORDINGS_FOLDER is None:
    hinted = [d for d in own_roots if OWN_DATASET_HINT in os.path.basename(d).lower()]
    search_in = hinted or own_roots
    extra_abs = os.path.abspath(EXTRA_ROOT) if EXTRA_ROOT else None
    best, best_n = None, 0
    for r in search_in:
        for root, dirs, files in os.walk(r):
            # Both datasets can end up nested under the SAME top-level folder
            # (e.g. /kaggle/input/datasets/<owner>/<slug>/...), so excluding
            # EXTRA_ROOT from `own_roots` is not enough - prune its subtree here too.
            if extra_abs and os.path.abspath(root).startswith(extra_abs):
                dirs[:] = []
                continue
            n = sum(1 for f in files if f.endswith(".wav"))
            if n > best_n:
                best, best_n = root, n
    RECORDINGS_FOLDER = best

if not SPREADSHEET_PATH or not RECORDINGS_FOLDER:
    print("\nTree of your (non-dialect) datasets:")
    for r in own_roots:
        for root, dirs, files in os.walk(r):
            d = root.count("/") - 2
            if d <= 3:
                print("  " * d + os.path.basename(root) + "/", f"({len(files)} files)")
    raise FileNotFoundError("Attach your recordings dataset, or fix the names in CONFIG.")

# hard guard: your recordings must never resolve inside the dialect dataset
if EXTRA_ROOT and os.path.abspath(RECORDINGS_FOLDER).startswith(os.path.abspath(EXTRA_ROOT)):
    raise RuntimeError(
        f"RECORDINGS_FOLDER resolved inside the dialect dataset ({RECORDINGS_FOLDER}). "
        "Set OWN_DATASET_HINT in CONFIG to match your own dataset's folder name."
    )

n_wav = len(glob.glob(RECORDINGS_FOLDER + "/*.wav"))
print("\nspreadsheet  :", SPREADSHEET_PATH)
print("your recordings:", RECORDINGS_FOLDER, f"({n_wav} wavs)")
print("dialect set  :", EXTRA_ROOT)
if n_wav < 600:
    print(f"\nNOTE: {n_wav} wavs, not 600. Check you attached the corrected set.")

## Base model + codec

In [ ]:
import os, sys, importlib
from huggingface_hub import snapshot_download

snapshot_download(ARABIC_BASE_REPO, local_dir=ARABIC_SNAPSHOT_DIR)
snapshot_download(SPARK_BASE_REPO, local_dir=SPARK_SNAPSHOT_DIR)
print("arabic repo:", sorted(os.listdir(ARABIC_SNAPSHOT_DIR)))

has_llm_subdir = os.path.exists(os.path.join(ARABIC_SNAPSHOT_DIR, "LLM", "config.json"))
LLM_DIR = os.path.join(ARABIC_SNAPSHOT_DIR, "LLM") if has_llm_subdir else ARABIC_SNAPSHOT_DIR
print("LLM_DIR   =", LLM_DIR)

repo_dir = os.path.abspath("Spark-TTS")
if not os.path.exists(os.path.join(repo_dir, "sparktts")):
    raise RuntimeError(f"sparktts not found in {repo_dir}. Re-run the install cell.")
for p in ["Spark-TTS", repo_dir]:
    if p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, repo_dir)
importlib.invalidate_caches()
from sparktts.models.audio_tokenizer import BiCodecTokenizer
print("sparktts imported.")

## Codec gate — do not skip

Encodes one of your real recordings and decodes it straight back. **Listen to both clips.** If
the round-trip does not sound like your recording, training will produce noise no matter what
else is correct.

In [ ]:
import glob, os, torch, numpy as np, librosa
from IPython.display import Audio as IPythonAudio, display

def roundtrip(tok, wav_path):
    """Encode a real recording, decode it straight back, score the similarity.
    ~1.0 = faithful; near 0 = the output is unrelated to the input, i.e. noise."""
    sr = tok.config["sample_rate"]
    orig, _ = librosa.load(wav_path, sr=sr, mono=True)
    g, s = tok.tokenize(wav_path)
    with torch.no_grad():
        rec = tok.detokenize(g.reshape(1, -1).to(tok.device), s.reshape(1, -1).to(tok.device))
    rec = np.squeeze(rec.detach().cpu().numpy() if isinstance(rec, torch.Tensor) else rec)
    n = min(len(orig), len(rec))
    if n < sr // 2:
        return 0.0, orig, rec
    a = librosa.power_to_db(librosa.feature.melspectrogram(y=orig[:n], sr=sr, n_mels=64)).flatten()
    b = librosa.power_to_db(librosa.feature.melspectrogram(y=rec[:n], sr=sr, n_mels=64)).flatten()
    m = min(len(a), len(b))
    return float(np.corrcoef(a[:m], b[:m])[0, 1]), orig, rec

probe = sorted(glob.glob(f"{RECORDINGS_FOLDER}/*.wav"))[0]
print("probe file:", probe)

candidates = []
if os.path.exists(os.path.join(ARABIC_SNAPSHOT_DIR, "BiCodec", "model.safetensors")):
    candidates.append(("arabic-bundled", ARABIC_SNAPSHOT_DIR))
candidates.append(("spark-original", SPARK_SNAPSHOT_DIR))

results = []
for name, d in candidates:
    try:
        tok = BiCodecTokenizer(d, "cuda")
        score, orig, rec = roundtrip(tok, probe)
        print(f"{name:16s} round-trip score = {score:.3f}")
        results.append((score, name, d, tok, orig, rec))
    except Exception as e:
        print(f"{name:16s} FAILED: {type(e).__name__}: {e}")

if not results:
    raise RuntimeError("No codec could be loaded at all.")

results.sort(reverse=True, key=lambda r: r[0])
best_score, best_name, CODEC_DIR, audio_tokenizer, orig, rec = results[0]
print("\nSELECTED CODEC:", best_name, f"(score {best_score:.3f})")

sr = audio_tokenizer.config["sample_rate"]
print("\nORIGINAL:")
display(IPythonAudio(orig, rate=sr))
print("ROUND-TRIP (this must sound like the original):")
display(IPythonAudio(rec, rate=sr))

if best_score < CODEC_MIN_SCORE:
    raise RuntimeError(
        f"Best round-trip score is only {best_score:.3f}. The codec cannot reproduce real "
        "audio, so every training token would be noise. Do NOT train until this passes."
    )
print(f"\nGate passed (>= {CODEC_MIN_SCORE}).")

## Build your dataset

In [ ]:
import os, re, glob, shutil
import numpy as np
import librosa, soundfile as sf
import openpyxl

# Fallback only - the corrected set has filename == ID, so this is not normally used.
CATEGORY_TO_PREFIX = {
    "Greetings & General": "GG", "Patient Registration": "PR", "Appointment Booking": "AB",
    "Rescheduling": "R", "Cancellation": "CAN", "Confirmations": "CNF",
    "Dental Procedures & Vocabulary": "DPV", "Patient Symptoms & Complaints": "PSC",
    "Doctor Instructions & Aftercare": "DIA", "Pricing & Payment": "PP",
    "Filler Words & Natural Speech": "FWN", "Numbers, Dates & Times": "NDT",
}

wb = openpyxl.load_workbook(SPREADSHEET_PATH, data_only=True)
ws = wb[SPREADSHEET_SHEET]
rows = [r for r in ws.iter_rows(min_row=2, values_only=True) if r[0]]
print(f"spreadsheet rows: {len(rows)}")

wav_stems = {os.path.splitext(f)[0] for f in os.listdir(RECORDINGS_FOLDER) if f.endswith(".wav")}
print(f"wav files       : {len(wav_stems)}")

pairs, unmatched, used, via_fallback = [], [], set(), 0
for r in rows:
    rid, cat = str(r[0]).strip(), str(r[1]).strip()
    text = str(r[2]).strip() if r[2] else ""
    stem = rid if rid in wav_stems else None
    if stem is None:
        prefix = CATEGORY_TO_PREFIX.get(cat)
        if prefix:
            cand = f"{prefix}-{rid.split('-')[-1]}"
            if cand in wav_stems:
                stem, via_fallback = cand, via_fallback + 1
    if stem and stem not in used and text:
        used.add(stem)
        pairs.append((stem, text))
    else:
        unmatched.append(rid)

print(f"matched         : {len(pairs)}  (direct: {len(pairs)-via_fallback}, fallback: {via_fallback})")
print(f"unmatched rows  : {len(unmatched)} {unmatched[:5]}")
print(f"wavs never used : {sorted(wav_stems - used)[:5]}")
if len(pairs) < len(wav_stems) * 0.95:
    raise RuntimeError("Matched far fewer rows than there are wavs - check the folder/spreadsheet.")

# Trim silence, resample to the codec rate, peak-normalise. Done once, up front.
os.makedirs(PREPPED_DIR, exist_ok=True)
target_sr = audio_tokenizer.config["sample_rate"]
kept, dropped = [], []
for stem, text in pairs:
    src = os.path.join(RECORDINGS_FOLDER, stem + ".wav")
    dst = os.path.join(PREPPED_DIR, stem + ".wav")
    y, _ = librosa.load(src, sr=target_sr, mono=True)
    yt, _ = librosa.effects.trim(y, top_db=TRIM_TOP_DB)
    if len(yt) < 0.4 * target_sr:
        dropped.append(stem)
        continue
    pad = int(EDGE_PAD_SEC * target_sr)
    yt = np.concatenate([np.zeros(pad, np.float32), yt, np.zeros(pad, np.float32)])
    peak = float(np.abs(yt).max())
    if peak > 0:
        yt = (yt / peak * 0.9).astype(np.float32)
    sf.write(dst, yt, target_sr)
    kept.append({"audio": dst, "text": text})

raw_secs = sum(librosa.get_duration(path=os.path.join(RECORDINGS_FOLDER, s + ".wav"))
               for s, _ in pairs)
new_secs = sum(librosa.get_duration(path=k["audio"]) for k in kept)
print(f"\nprepared: {len(kept)} clips (dropped {len(dropped)} too short)")
print(f"audio    : {raw_secs/60:.1f} min -> {new_secs/60:.1f} min after trimming")

In [ ]:
import re
DIACRITICS = re.compile(r"[\u064B-\u0652\u0670]")

texts = [k["text"] for k in kept]
with_marks = sum(1 for t in texts if DIACRITICS.search(t))
chars = sum(len(t) for t in texts)
marks = sum(len(DIACRITICS.findall(t)) for t in texts)

print(f"sentences with any diacritic: {with_marks}/{len(texts)} = {with_marks/len(texts):.1%}")
print(f"diacritic characters        : {marks}/{chars} = {marks/chars:.2%}")
print("\nsample:", texts[0])
if marks / max(chars, 1) < 0.05:
    print("\nYour text is effectively undiacritized, while MrEzzat/Spark_TTS_Arabic was trained")
    print("on fully-diacritized MSA (ArVoice). This mismatch is the most likely remaining cause")
    print("of weak pronunciation on words outside your training set. It is a data question,")
    print("not a bug - train first, listen, then decide whether diacritizing is worth it.")

## Tokenize your recordings

In [ ]:
import json, torch

def audio_to_training_text(wav_path, text):
    g, s = audio_tokenizer.tokenize(wav_path)
    g_ids = g.reshape(-1).tolist()
    s_ids = s.reshape(-1).tolist()
    return "".join([
        "<|task_tts|>", "<|start_content|>", text.strip(), "<|end_content|>",
        "<|start_global_token|>", "".join(f"<|bicodec_global_{int(i)}|>" for i in g_ids),
        "<|end_global_token|>",
        "<|start_semantic_token|>", "".join(f"<|bicodec_semantic_{int(i)}|>" for i in s_ids),
        "<|end_semantic_token|>", "<|im_end|>",
    ]), len(g_ids), len(s_ids)

print("tokenizer helper ready.")

In [ ]:
import statistics

records, kept_ok, gl, sl = [], [], [], []
with open(OWN_JSONL, "w", encoding="utf-8") as f:
    for n, item in enumerate(kept):
        try:
            txt, ng, ns = audio_to_training_text(item["audio"], item["text"])
        except Exception as e:
            print(f"  skipped {item['audio']}: {type(e).__name__}: {e}")
            continue
        records.append(txt); kept_ok.append(item); gl.append(ng); sl.append(ns)
        f.write(json.dumps({"text": txt}, ensure_ascii=False) + "\n")
        if (n + 1) % 100 == 0:
            print(f"  {n+1}/{len(kept)}", flush=True)

print(f"\ntokenized {len(records)} clips -> {OWN_JSONL}")
print(f"semantic tokens per clip: min={min(sl)} mean={statistics.mean(sl):.0f} max={max(sl)}")
print(f"semantic tokens/second  : ~{statistics.mean(sl)/(new_secs/len(records)):.0f}")

## Sanity check — listen before training

Decodes the exact tokens the model will train on, back into audio.

In [ ]:
import random, numpy as np
from IPython.display import Audio as IPythonAudio, display

def decode_training_text(t):
    g = [int(x) for x in re.findall(r"<\|bicodec_global_(\d+)\|>", t)]
    s = [int(x) for x in re.findall(r"<\|bicodec_semantic_(\d+)\|>", t)]
    if not g or not s:
        raise ValueError("no audio tokens in this record")
    gi = torch.tensor(g, dtype=torch.long, device="cuda").reshape(1, -1)
    si = torch.tensor(s, dtype=torch.long, device="cuda").reshape(1, -1)
    with torch.no_grad():
        wav = audio_tokenizer.detokenize(gi, si)
    return np.squeeze(wav.detach().cpu().numpy() if isinstance(wav, torch.Tensor) else wav)

sr = audio_tokenizer.config["sample_rate"]
for i in [0] + random.sample(range(len(records)), k=min(2, len(records))):
    print(f"--- record {i} ---")
    print(kept_ok[i]["text"])
    print("original:")
    display(IPythonAudio(kept_ok[i]["audio"]))
    print("decoded from the training tokens:")
    display(IPythonAudio(decode_training_text(records[i]), rate=sr))

print("\nLISTEN. If the decoded audio is not recognisably the same speech, STOP.")
print("Also check the text matches what you hear - that is what the shift fix was about.")

## Egyptian dialect data — Kaggle dataset

Replaces NileTTS. Mounted read-only by Kaggle, so nothing is downloaded and nothing is written
to your 20 GB working directory.

The first cell only **inspects** the mount and prints what it found — file types, columns, a few
sample rows. Read that output before running the loader, so you can correct the column names in
the loader if the auto-detection guessed wrong.

In [ ]:
import os, glob

if not USE_EXTRA_DIALECT or EXTRA_ROOT is None:
    print("USE_EXTRA_DIALECT is False - skipping.")
else:
    print("dialect dataset root:", EXTRA_ROOT)

    print("\n--- directory tree (2 levels) ---")
    for root, dirs, files in os.walk(EXTRA_ROOT):
        depth = root.replace(EXTRA_ROOT, "").count(os.sep)
        if depth > 2:
            dirs[:] = []
            continue
        print("  " * depth + os.path.basename(root) + "/", f"({len(files)} files)")
        for f in sorted(files)[:4]:
            print("  " * (depth + 1) + f)
        if len(files) > 4:
            print("  " * (depth + 1) + f"... and {len(files)-4} more")

    exts = {}
    for root, dirs, files in os.walk(EXTRA_ROOT):
        for f in files:
            exts[os.path.splitext(f)[1].lower()] = exts.get(os.path.splitext(f)[1].lower(), 0) + 1
    print("\nfile types:", dict(sorted(exts.items(), key=lambda kv: -kv[1])))

    tables = [p for p in glob.glob(EXTRA_ROOT + "/**/*", recursive=True)
              if os.path.splitext(p)[1].lower() in (".csv", ".tsv", ".parquet", ".json", ".jsonl")]
    print("\nmetadata candidates:", [os.path.basename(t) for t in tables[:10]])

    import pandas as pd
    for t in tables[:3]:
        try:
            if t.endswith(".parquet"):
                df_peek = pd.read_parquet(t)
            elif t.endswith((".json", ".jsonl")):
                df_peek = pd.read_json(t, lines=t.endswith(".jsonl"))
            else:
                df_peek = pd.read_csv(t, sep=None, engine="python", nrows=200)
            print(f"\n--- {os.path.basename(t)} --- shape={df_peek.shape}")
            print("columns:", list(df_peek.columns))
            print(df_peek.head(3).to_string()[:800])
        except Exception as e:
            print(f"\n--- {os.path.basename(t)} --- could not read: {type(e).__name__}: {e}")

In [ ]:
import os, glob, json, statistics
import pandas as pd
import numpy as np
import librosa, soundfile as sf

if not USE_EXTRA_DIALECT or EXTRA_ROOT is None:
    print("Skipping extra dialect data.")
else:
    # ---- pick the metadata table -------------------------------------------------
    tables = [p for p in glob.glob(EXTRA_ROOT + "/**/*", recursive=True)
              if os.path.splitext(p)[1].lower() in (".csv", ".tsv", ".parquet", ".json", ".jsonl")]
    if not tables:
        raise FileNotFoundError("No csv/parquet/json metadata found in the dataset.")
    tables.sort(key=lambda p: -os.path.getsize(p))
    META = tables[0]
    print("using metadata:", META)

    if META.endswith(".parquet"):
        df = pd.read_parquet(META)
    elif META.endswith((".json", ".jsonl")):
        df = pd.read_json(META, lines=META.endswith(".jsonl"))
    else:
        df = pd.read_csv(META, sep=None, engine="python")
    print("shape:", df.shape, "| columns:", list(df.columns))

    # ---- identify the audio-path and text columns --------------------------------
    def pick(cands, kws):
        for kw in kws:
            for c in cands:
                if kw in c.lower():
                    return c
        return None

    cols = list(df.columns)
    AUDIO_COL = pick(cols, ["audio", "path", "file", "wav", "clip"])
    TEXT_COL  = pick(cols, ["text", "sentence", "transcript", "label", "caption"])
    if AUDIO_COL is None or TEXT_COL is None:
        raise RuntimeError(
            f"Could not identify columns automatically. Columns are {cols}. "
            "Set AUDIO_COL and TEXT_COL by hand and re-run this cell."
        )
    print(f"audio column: {AUDIO_COL!r} | text column: {TEXT_COL!r}")

    # ---- resolve audio paths -----------------------------------------------------
    audio_index = {}
    for p in glob.glob(EXTRA_ROOT + "/**/*", recursive=True):
        if os.path.splitext(p)[1].lower() in (".wav", ".mp3", ".flac", ".m4a", ".ogg"):
            audio_index.setdefault(os.path.basename(p), p)
    print("audio files on disk:", len(audio_index))

    def resolve(v):
        v = str(v).strip()
        if os.path.isabs(v) and os.path.exists(v):
            return v
        cand = os.path.join(EXTRA_ROOT, v)
        if os.path.exists(cand):
            return cand
        return audio_index.get(os.path.basename(v))

    # ---- filter and tokenize directly from the read-only mount -------------------
    target_sr = audio_tokenizer.config["sample_rate"]
    n_written = n_seen = 0
    reasons = {"no_file": 0, "short_text": 0, "duration": 0, "error": 0}
    tmp = "/tmp/_extra.wav"

    with open(EXTRA_JSONL, "w", encoding="utf-8") as out:
        for _, row in df.iterrows():
            if n_written >= EXTRA_MAX_CLIPS:
                break
            n_seen += 1
            text = str(row[TEXT_COL]).strip()
            if len(text) < EXTRA_MIN_CHARS:
                reasons["short_text"] += 1
                continue
            src = resolve(row[AUDIO_COL])
            if not src:
                reasons["no_file"] += 1
                continue
            try:
                y, _ = librosa.load(src, sr=target_sr, mono=True)
                yt, _ = librosa.effects.trim(y, top_db=TRIM_TOP_DB)
                dur = len(yt) / target_sr
                if not (EXTRA_MIN_SEC <= dur <= EXTRA_MAX_SEC):
                    reasons["duration"] += 1
                    continue
                pad = int(EDGE_PAD_SEC * target_sr)
                yt = np.concatenate([np.zeros(pad, np.float32), yt, np.zeros(pad, np.float32)])
                peak = float(np.abs(yt).max())
                if peak > 0:
                    yt = (yt / peak * 0.9).astype(np.float32)
                sf.write(tmp, yt, target_sr)
                txt, _, _ = audio_to_training_text(tmp, text)
                out.write(json.dumps({"text": txt}, ensure_ascii=False) + "\n")
                n_written += 1
                if n_written % 100 == 0:
                    print(f"  {n_written}/{EXTRA_MAX_CLIPS} kept "
                          f"(scanned {n_seen})", flush=True)
            except Exception:
                reasons["error"] += 1
                continue

    print(f"\nkept {n_written} clips from {n_seen} rows scanned -> {EXTRA_JSONL}")
    print("rejected:", reasons)

## Mix into one training set

In [ ]:
import os
from datasets import load_dataset, concatenate_datasets

own = load_dataset("json", data_files=OWN_JSONL, split="train")
parts = [own] * OWN_OVERSAMPLE
print(f"your clips  : {len(own)} x{OWN_OVERSAMPLE} = {len(own)*OWN_OVERSAMPLE}")

if USE_EXTRA_DIALECT and os.path.exists(EXTRA_JSONL) and os.path.getsize(EXTRA_JSONL) > 0:
    extra = load_dataset("json", data_files=EXTRA_JSONL, split="train")
    parts.append(extra)
    print(f"dialect data: {len(extra)}")
    share = len(extra) / (len(own)*OWN_OVERSAMPLE + len(extra))
    print(f"dialect share of the mix: {share:.0%}")
    if share > 0.6:
        print("  NOTE: the noisy YouTube data now dominates. Consider raising OWN_OVERSAMPLE")
        print("  or lowering EXTRA_MAX_CLIPS if your voice comes out muddy.")
else:
    print("dialect data: not included")

train_dataset = concatenate_datasets(parts).shuffle(seed=3407)
print("\ntraining set:", len(train_dataset), "examples")

## Fine-tune

LoRA on top of the Arabic checkpoint. Full fine-tuning on this little data would destroy the
Arabic ability you are building on.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

tokenizer = AutoTokenizer.from_pretrained(LLM_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# fp32, not fp16 - this model emits zero audio tokens in half precision.
model = AutoModelForCausalLM.from_pretrained(LLM_DIR, torch_dtype=torch.float32).to("cuda")

probe = ["<|task_tts|>", "<|start_global_token|>", "<|bicodec_global_0|>",
         "<|bicodec_semantic_0|>", "<|im_end|>"]
split = [t for t in probe if len(tokenizer.encode(t, add_special_tokens=False)) != 1]
for t in probe:
    n = len(tokenizer.encode(t, add_special_tokens=False))
    print(f"{t:28s} -> {n} token(s) {'OK' if n == 1 else 'SPLIT'}")
if split:
    raise RuntimeError(
        f"These are not single tokens: {split}. The tokenizer has no audio-token vocabulary, "
        "so training would see subword fragments. This base checkpoint cannot be used."
    )

lora = LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

def to_ids(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_SEQ_LEN)

ds_ids = train_dataset.map(to_ids, batched=True, remove_columns=train_dataset.column_names,
                           desc="text -> input_ids")

lens = [len(x) for x in ds_ids["input_ids"][:500]]
n_trunc = sum(1 for l in lens if l >= MAX_SEQ_LEN)
print(f"sequence length: mean {sum(lens)/len(lens):.0f}, max {max(lens)}, "
      f"{n_trunc}/{len(lens)} hit the {MAX_SEQ_LEN} cap")
if n_trunc > len(lens) * 0.05:
    print("  WARNING: many sequences truncated - their audio tokens are being cut off.")
    print("  Lower EXTRA_MAX_SEC or raise MAX_SEQ_LEN.")

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

model.config.use_cache = False
model.enable_input_require_grads()

args = TrainingArguments(
    output_dir=OUTPUT_MODEL_ID.split("/")[-1],
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_steps=10,
    lr_scheduler_type="linear",
    weight_decay=0.001,
    optim="adamw_torch",
    fp16=False, bf16=False,
    gradient_checkpointing=True,
    logging_steps=25,
    save_strategy="no",
    report_to="none",
    seed=3407,
)

trainer = Trainer(model=model, args=args, train_dataset=ds_ids, data_collator=collator)
stats = trainer.train()
print(stats)

hist = [h["loss"] for h in trainer.state.log_history if "loss" in h]
print(f"\nloss {hist[0]:.3f} -> {hist[-1]:.3f}")
if len(hist) > 6:
    early, late = sum(hist[:3])/3, sum(hist[-3:])/3
    print(f"first 3 logs mean {early:.3f} | last 3 logs mean {late:.3f}")
    if late > early * 0.95:
        print("Loss plateaued early - more epochs alone will not help. Consider LoRA r=32.")

In [ ]:
from huggingface_hub import HfApi

local_dir = OUTPUT_MODEL_ID.split("/")[-1] + "-final"
model.save_pretrained(local_dir)
tokenizer.save_pretrained(local_dir)
print("saved locally to", local_dir)

model.push_to_hub(OUTPUT_MODEL_ID, private=True, token=HF_TOKEN)
tokenizer.push_to_hub(OUTPUT_MODEL_ID, private=True, token=HF_TOKEN)
print("pushed to", OUTPUT_MODEL_ID)

for f in HfApi(token=HF_TOKEN).list_repo_files(OUTPUT_MODEL_ID, repo_type="model"):
    print("  -", f)

## Test — reference-conditioned

Supplies real speaker tokens from one of your recordings rather than making the model invent
them. This is how Spark-TTS cloning is designed to work, and it isolates content quality from
speaker quality.

In [ ]:
import time, glob, numpy as np
from IPython.display import Audio as IPythonAudio, display

model.config.use_cache = True
if getattr(model, "generation_config", None) is not None:
    model.generation_config.use_cache = True
model.eval()

EOS_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")

ref_wav = sorted(glob.glob(f"{PREPPED_DIR}/*.wav"))[0]
g_ref, _ = audio_tokenizer.tokenize(ref_wav)
g_list = [int(x) for x in g_ref.reshape(-1).tolist()]
global_str = "".join(f"<|bicodec_global_{i}|>" for i in g_list)
print("reference:", ref_wav, f"({len(g_list)} global tokens)")

def clone(text, max_new_tokens=MAX_GEN_TOKENS, do_sample=False, temperature=0.8):
    prompt = ("<|task_tts|><|start_content|>" + text.strip() + "<|end_content|>"
              "<|start_global_token|>" + global_str + "<|end_global_token|>"
              "<|start_semantic_token|>")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    kw = dict(do_sample=True, temperature=temperature, top_p=0.95) if do_sample else dict(do_sample=False)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True,
                             eos_token_id=EOS_ID,
                             pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id, **kw)
    new = out[0][inputs["input_ids"].shape[1]:]
    dec = tokenizer.decode(new, skip_special_tokens=False)
    s = [int(x) for x in re.findall(r"<\|bicodec_semantic_(\d+)\|>", dec)]
    hit = "hit cap" if len(new) >= max_new_tokens else "stopped early"
    print(f"  {len(new)} tokens in {time.time()-t0:.0f}s ({hit}) -> {len(s)} semantic")
    if not s:
        print("  no semantic tokens produced")
        return None
    gi = torch.tensor(g_list, dtype=torch.long, device="cuda").reshape(1, -1)
    si = torch.tensor(s, dtype=torch.long, device="cuda").reshape(1, -1)
    with torch.no_grad():
        wav = audio_tokenizer.detokenize(gi, si)
    wav = np.squeeze(wav.detach().cpu().numpy() if isinstance(wav, torch.Tensor) else wav)
    sr = audio_tokenizer.config["sample_rate"]
    print(f"  {len(wav)/sr:.2f}s of audio (~{len(s)/max(len(wav)/sr,1e-6):.0f} sem tokens/sec)")
    display(IPythonAudio(wav, rate=sr))
    return wav

for t in TEST_SENTENCES:
    print("\n" + t)
    clone(t)

In [ ]:
# Memorization check: a sentence the model definitely trained on.
# If this is clear but novel sentences are not, the gap is generalization, not capacity.
print("TRAINED SENTENCE:", kept_ok[0]["text"])
print("original recording:")
display(IPythonAudio(kept_ok[0]["audio"]))
print("model output:")
clone(kept_ok[0]["text"])